# Anomaly Explanation with LLMs

## Overview
This notebook demonstrates a pipeline for identifying, contextualizing, and explaining anomalies in World Bank Groups Corporate scorecard (CSC) time series data using a Large Language Model (LLM). The goal is to automatically generate verifiable explanations for significant data deviations, classifying them as external drivers, data errors, or measurement system updates.

## Rationale
Detecting and understanding anomalies in World Bank Groups Corporate scorecard time series is crucial for data quality assurance and policy analysis. Manual investigation of numerous anomalies is time-consuming. This notebook leverages LLMs to automate this process, providing structured explanations grounded in historical events and data characteristics, thereby accelerating diagnostic workflows.

## Methods
1.  **Data Loading and Preprocessing**: World Bank Groups Corporate scorecard data, including pre-calculated anomaly scores, country, and series metadata, is loaded and prepared for analysis.
2.  **Anomaly Identification**: The notebook filters for the most significant anomalies based on combined Z-scores and the total number of outlier indicators for a given time series.
3.  **Context Generation**: For each identified anomaly, a time-windowed context (a snippet of the time series data around the anomaly) is generated. This context includes the indicator values and imputation status, facilitating LLM analysis.
4.  **LLM Prompting**: A structured prompt, including system instructions, user task, analysis rules, and the generated time-series context, is prepared for an LLM. The LLM is instructed to identify anomaly windows, classify their causes, and provide verifiable explanations with confidence scores and evidence sources.
5.  **LLM Inference (Batch Processing)**: The prepared prompts are sent to an LLM (e.g., GPT-4.1-mini) for batch inference, generating anomaly explanations in a predefined JSON format.
6.  **Output Analysis**: The LLM responses are parsed and compiled into a DataFrame, allowing for aggregated analysis of the generated anomaly explanations, classifications, and confidence levels.

## Data Loading and Preprocessing

In [ ]:
import json
import pandas as pd
import os
from jinja2 import Template

In [ ]:
# # Setup directories

# ## Create project dir
# !mkdir /content/anomaly-explanation

# ## Input directory for raw files from the macroanomaly package
# !mkdir /content/anomaly-explanation/raw-input

# !mkdir /content/anomaly-explanation/llm-input

# !mkdir /content/anomaly-explanation/llm-output


In [ ]:
!ln -s /content/drive/MyDrive/WBG/Anomaly\ Explanation /content/anomaly-explanation

In [ ]:
# Set the value as the basename of the file
raw_fname = "CSC_TOP_ANOMALIES_2026-02-06"

PROJECT_NAME = os.path.basename(raw_fname)

PROJECT_DIR = f"/content/anomaly-explanation/{PROJECT_NAME}"
LLM_INPUT_DIR = f"{PROJECT_DIR}/llm-input"
LLM_OUTPUT_DIR = f"{PROJECT_DIR}/llm-output"
# RAW_INPUT_DIR = f"{PROJECT_DIR}/raw-input"
PROCESSED_DIR = f"{PROJECT_DIR}/processed"

# Create project dirs
# os.makedirs(f"/content/{PROJECT_NAME}/raw-input", exist_ok=True)
os.makedirs(LLM_INPUT_DIR, exist_ok=True)
os.makedirs(LLM_OUTPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
# Define the column variables

INDICATOR_CODE_COL =  "INDICATOR"  # "Indicator.Code"
INDICATOR_NAME_COL = "INDICATOR_LABEL"  # "Indicator.Name
COUNTRY_CODE_COL = "REF_AREA"  # "Country.Code"
COUNTRY_NAME_COL = "REF_AREA_LABEL"  # "Country.Name
YEAR_COL = "YEAR"  # "Year"
VALUE_COL = "VALUE"  # "Value"
IMPUTED_COL = "Imputed"

In [ ]:
full_df = pd.read_csv(f"{PROJECT_DIR}/WB_CSC_WIDEF.csv")
full_df.head()

,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,SEX,SEX_LABEL,AGE,AGE_LABEL,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,A,Annual,ABW,Aruba,WB_CSC_EN_ATM_GHGT_GT_CE,Total greenhouse gas emissions,_Z,Not applicable,_Z,Not applicable,...,4.764612e+05,4.941140e+05,4.671886e+05,5.020768e+05,5.758223e+05,4.947307e+05,4.791918e+05,4.966828e+05,NaN,NaN
1,A,Annual,ABW,Aruba,WB_CSC_ER_LND_HEAL,Hectares of key ecosystems (globally),_Z,Not applicable,_Z,Not applicable,...,NaN,NaN,NaN,NaN,NaN,1.402161e+04,NaN,NaN,NaN,NaN
2,A,Annual,ABW,Aruba,WB_CSC_SH_H2O_BASW_ZS,Percentage of people with access to basic drin...,_Z,Not applicable,_Z,Not applicable,...,9.790000e+01,9.790000e+01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A,Annual,ABW,Aruba,WB_CSC_SH_STA_BASS_ZS,Percentage of people with access to basic sani...,_Z,Not applicable,_Z,Not applicable,...,9.860000e+01,9.860000e+01,9.860000e+01,9.870000e+01,9.870000e+01,9.880000e+01,9.880000e+01,9.880000e+01,NaN,NaN
4,A,Annual,AFE,Africa Eastern and Southern,WB_CSC_EN_ATM_GHGT_GT_CE,Total greenhouse gas emissions,_Z,Not applicable,_Z,Not applicable,...,1.772779e+09,1.775147e+09,1.798251e+09,1.820748e+09,1.855796e+09,1.799098e+09,1.823251e+09,1.828657e+09,NaN,NaN


In [ ]:
# Identify non-year columns (id_vars) and year columns (value_vars)
# Assuming year columns are those that can be converted to integers
non_year_cols = []
year_cols = []

for col in full_df.columns:
    try:
        int(col)
        year_cols.append(col)
    except ValueError:
        non_year_cols.append(col)

# Perform the melt operation
long_df = pd.melt(full_df, id_vars=non_year_cols, value_vars=year_cols, var_name=YEAR_COL, value_name=VALUE_COL)

# Convert the 'YEAR_COL' back to integer type if needed
long_df[YEAR_COL] = pd.to_numeric(long_df[YEAR_COL], errors='coerce').astype('Int64')

# Display the first few rows of the new long format DataFrame
display(long_df.head())

,FREQ,FREQ_LABEL,REF_AREA,REF_AREA_LABEL,INDICATOR,INDICATOR_LABEL,SEX,SEX_LABEL,AGE,AGE_LABEL,...,DATABASE_ID,DATABASE_ID_LABEL,UNIT_MULT,UNIT_MULT_LABEL,OBS_STATUS,OBS_STATUS_LABEL,OBS_CONF,OBS_CONF_LABEL,YEAR,VALUE
0,A,Annual,ABW,Aruba,WB_CSC_EN_ATM_GHGT_GT_CE,Total greenhouse gas emissions,_Z,Not applicable,_Z,Not applicable,...,WB_CSC,World Bank Group Corporate Scorecard,0,Units,A,Normal value,PU,Public,1998,NaN
1,A,Annual,ABW,Aruba,WB_CSC_ER_LND_HEAL,Hectares of key ecosystems (globally),_Z,Not applicable,_Z,Not applicable,...,WB_CSC,World Bank Group Corporate Scorecard,0,Units,A,Normal value,PU,Public,1998,NaN
2,A,Annual,ABW,Aruba,WB_CSC_SH_H2O_BASW_ZS,Percentage of people with access to basic drin...,_Z,Not applicable,_Z,Not applicable,...,WB_CSC,World Bank Group Corporate Scorecard,0,Units,A,Normal value,PU,Public,1998,NaN
3,A,Annual,ABW,Aruba,WB_CSC_SH_STA_BASS_ZS,Percentage of people with access to basic sani...,_Z,Not applicable,_Z,Not applicable,...,WB_CSC,World Bank Group Corporate Scorecard,0,Units,A,Normal value,PU,Public,1998,NaN
4,A,Annual,AFE,Africa Eastern and Southern,WB_CSC_EN_ATM_GHGT_GT_CE,Total greenhouse gas emissions,_Z,Not applicable,_Z,Not applicable,...,WB_CSC,World Bank Group Corporate Scorecard,0,Units,A,Normal value,PU,Public,1998,NaN


In [ ]:
input_fname = f"{PROJECT_DIR}/{raw_fname}.CSV"

raw_df = pd.read_csv(input_fname)
print(raw_df.shape)
raw_df["absZscore"] = raw_df["Zscore"].abs()

common_cols = [i for i in raw_df.columns if i in long_df.columns]
o_df = long_df.merge(raw_df, on=common_cols, how="left")

raw_df = raw_df[[INDICATOR_CODE_COL, COUNTRY_CODE_COL]].drop_duplicates().merge(o_df, on=[INDICATOR_CODE_COL, COUNTRY_CODE_COL], how="left")[raw_df.columns]

raw_df["Imputed"] = raw_df["Imputed"].fillna(False)

# Index properly for faster query when generating the context
source_df = raw_df.sort_values([INDICATOR_CODE_COL, COUNTRY_CODE_COL, YEAR_COL])
print(source_df.shape)
source_df = source_df.set_index([INDICATOR_CODE_COL, COUNTRY_CODE_COL]).sort_index()
print(source_df.shape)
print(raw_df.shape)
source_df.head()

(354, 47)
(9558, 48)
(9558, 46)
(9558, 48)


/tmp/ipython-input-1380101741.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  raw_df["Imputed"] = raw_df["Imputed"].fillna(False)


REF_AREA_LABEL  YEAR  VALUE  Zscore FREQ  \
INDICATOR             REF_AREA                                            
WB_CSC_EG_ELC_ACCS_ZS ALB             Albania  1998    NaN     NaN    A   
                      ALB             Albania  1999    NaN     NaN    A   
                      ALB             Albania  2000   99.4     NaN    A   
                      ALB             Albania  2001   99.4     NaN    A   
                      ALB             Albania  2002   99.4     NaN    A   

                               FREQ_LABEL  \
INDICATOR             REF_AREA              
WB_CSC_EG_ELC_ACCS_ZS ALB          Annual   
                      ALB          Annual   
                      ALB          Annual   
                      ALB          Annual   
                      ALB          Annual   

                                                                  INDICATOR_LABEL  \
INDICATOR             REF_AREA                                                      
WB_CSC_EG_ELC_ACCS_ZS ALB       Percentage of population with access to electr...   
                      ALB       Percentage of population with access to electr...   
                      ALB       Percentage of population with access to electr...   
                      ALB       Percentage of population with access to electr...   
                      ALB       Percentage of population with access to electr...   

                               SEX       SEX_LABEL AGE  ... mean.change_capa  \
INDICATOR             REF_AREA                          ...                    
WB_CSC_EG_ELC_ACCS_ZS ALB       _Z  Not applicable  _Z  ...              NaN   
                      ALB       _Z  Not applicable  _Z  ...              NaN   
                      ALB       _Z  Not applicable  _Z  ...              NaN   
                      ALB       _Z  Not applicable  _Z  ...              NaN   
                      ALB       _Z  Not applicable  _Z  ...              NaN   

                               variance.change_capa test.statistic_capa  \
INDICATOR             REF_AREA                                            
WB_CSC_EG_ELC_ACCS_ZS ALB                       NaN                 NaN   
                      ALB                       NaN                 NaN   
                      ALB                       NaN                 NaN   
                      ALB                       NaN                 NaN   
                      ALB                       NaN                 NaN   

                               outlier_score_outliertree  \
INDICATOR             REF_AREA                             
WB_CSC_EG_ELC_ACCS_ZS ALB                            NaN   
                      ALB                            NaN   
                      ALB                            NaN   
                      ALB                            NaN   
                      ALB                            NaN   

                               outlier_indicator_outliertree absZscore_zscore  \
INDICATOR             REF_AREA                                                  
WB_CSC_EG_ELC_ACCS_ZS ALB                                NaN              NaN   
                      ALB                                NaN              NaN   
                      ALB                                NaN              NaN   
                      ALB                                NaN              NaN   
                      ALB                                NaN              NaN   

                               rankZscore_zscore outlier_indicator_zscore  \
INDICATOR             REF_AREA                                              
WB_CSC_EG_ELC_ACCS_ZS ALB                    NaN                      NaN   
                      ALB                    NaN                      NaN   
                      ALB                    NaN                      NaN   
                      ALB                    NaN                      NaN   
                      ALB                    NaN       

In [ ]:
country_code_names = raw_df[[COUNTRY_CODE_COL, COUNTRY_NAME_COL]].drop_duplicates().set_index(COUNTRY_CODE_COL).to_dict()[COUNTRY_NAME_COL]
series_code_names = raw_df[[INDICATOR_CODE_COL, INDICATOR_NAME_COL]].drop_duplicates().set_index(INDICATOR_CODE_COL).to_dict()[INDICATOR_NAME_COL]

len(country_code_names), len(series_code_names)

(168, 23)

In [ ]:
df = raw_df.sort_values(["outlier_indicator_total", "absZscore"], ascending=False)
df.head()

,REF_AREA,REF_AREA_LABEL,INDICATOR,YEAR,VALUE,Zscore,FREQ,FREQ_LABEL,INDICATOR_LABEL,SEX,...,mean.change_capa,variance.change_capa,test.statistic_capa,outlier_score_outliertree,outlier_indicator_outliertree,absZscore_zscore,rankZscore_zscore,outlier_indicator_zscore,outlier_indicator_total,absZscore
13,ISL,Iceland,WB_CSC_SI_POV_UMIC,2011,1.0,4.944843,A,Annual,Percentage of global population living in pove...,_Z,...,NaN,NaN,NaN,0.0,0.0,4.944843,1.0,1.0,3.0,4.944843
39,MYS,Malaysia,WB_CSC_EN_ATM_HAZA,2010,0.2,4.938949,A,Annual,Percentage of people exposed to hazardous air ...,_Z,...,NaN,NaN,NaN,0.0,0.0,4.938949,2.0,1.0,3.0,4.938949
72,STP,Sao Tome and Principe,WB_CSC_EN_ATM_HAZA,2016,0.2,4.937519,A,Annual,Percentage of people exposed to hazardous air ...,_Z,...,NaN,NaN,NaN,0.0,0.0,4.937519,3.0,1.0,3.0,4.937519
95,MNE,Montenegro,WB_CSC_EN_ATM_HAZA,2012,1.7,4.934878,A,Annual,Percentage of people exposed to hazardous air ...,_Z,...,NaN,NaN,NaN,0.0,0.0,4.934878,4.0,1.0,3.0,4.934878
122,LKA,Sri Lanka,WB_CSC_EN_ATM_HAZA,2012,11.2,4.934612,A,Annual,Percentage of people exposed to hazardous air ...,_Z,...,NaN,NaN,NaN,0.0,0.0,4.934612,5.0,1.0,3.0,4.934612


## Anomaly Identification

In [ ]:
_ddf = df[(df["outlier_indicator_total"] >= 3) & (~df["Imputed"])]
ordered_df = _ddf.groupby([INDICATOR_CODE_COL, COUNTRY_CODE_COL])["absZscore"].sum().sort_values(ascending=False).reset_index()
print(ordered_df.shape[0])

max_shortlist = 10_000

if ordered_df.shape[0] > max_shortlist:
    raise ValueError(f"Max shortlist size exceeded: {max_shortlist}")

shortlist = ordered_df.head(max_shortlist)
shortlist

354


,INDICATOR,REF_AREA,absZscore
0,WB_CSC_SI_POV_UMIC,ISL,4.944843
1,WB_CSC_EN_ATM_HAZA,MYS,4.938949
2,WB_CSC_EN_ATM_HAZA,STP,4.937519
3,WB_CSC_EN_ATM_HAZA,MNE,4.934878
4,WB_CSC_EN_ATM_HAZA,LKA,4.934612
...,...,...,...
349,WB_CSC_SL_EMP_WORK_ZS,SYR,3.502684
350,WB_CSC_SH_H2O_BASW_ZS_CC,BFA,3.502356
351,WB_CSC_SH_H2O_BASW_ZS,BFA,3.502356
352,WB_CSC_IT_NET_USER_ZS,TUV,3.501894


In [ ]:
# # Create context for the LLM
# # - Find the years where anomalies have been detected for a series in a given geography.
# # - For each of these years, add a window of N years.
# # - Find contiguous anomalies within N-years of each other and pack them into one context.

# # country = "VEN"
# # indicator_code = "MS.MIL.XPND.CN"

# # country = "SEN"
# # indicator_code = "NY.GDP.DISC.KN"

# # country = "KWT"
# # indicator_code = "SP.POP.GROW"

# # country = "KWT"
# # indicator_code = "NY.ADJ.SVNG.GN.ZS"

# # country = "CYM"
# # indicator_code = "EN.GHG.CH4.WA.MT.CE.AR5"

# # country = "HND"
# # indicator_code = "SH.DTH.1519"

# country = "ISL"
# indicator_code = "SI.POV.NAHC"

# _df = source_df.loc[(indicator_code, country)].sort_values("Year").reset_index()
# _df

## Context Generation

In [ ]:
# Define the function to create the context payload

INDICATOR_CODE_COL =  "INDICATOR"  # "Indicator.Code"
INDICATOR_NAME_COL = "INDICATOR_LABEL"  # "Indicator.Name
COUNTRY_CODE_COL = "REF_AREA"  # "Country.Code"
COUNTRY_NAME_COL = "REF_AREA_LABEL"  # "Country.Name
YEAR_COL = "YEAR"  # "Year"
VALUE_COL = "VALUE"  # "Value"
IMPUTED_COL = "Imputed"


def extract_anomaly_contexts(df, period_window: int = 3, outlier_indicator_total: int = 3):
    assert df[INDICATOR_CODE_COL].nunique() == 1, f"{INDICATOR_CODE_COL} must be unique"
    assert df[COUNTRY_CODE_COL].nunique() == 1, f"{COUNTRY_CODE_COL} must be unique"

    # Ensure the required columns are present.
    for col in [VALUE_COL, YEAR_COL, IMPUTED_COL]:
        assert col in df.columns, f"{col} must be present in the dataframe"

    country = country_code_names[df[COUNTRY_CODE_COL].iloc[0]]
    indicator_name = series_code_names[df[INDICATOR_CODE_COL].iloc[0]]

    df = df.sort_values(YEAR_COL)

    # Create context for the LLM
    df = df.sort_values(YEAR_COL)
    min_year = df[YEAR_COL].min()
    max_year = df[YEAR_COL].max()

    # Find the years where anomalies have been detected for a series in a given geography.
    years = df.loc[(df["outlier_indicator_total"] >= outlier_indicator_total) & (~df[IMPUTED_COL]), YEAR_COL]

    low_years = years - period_window
    high_years = years + period_window

    periods = [list(range(max(min_year, i), min(j, max_year) + 1)) for i, j in zip(low_years, high_years)]

    valid_periods = []
    tmp_valid_period = set(periods[0])

    for period in periods[1:]:
        if tmp_valid_period.intersection(period):
            tmp_valid_period = tmp_valid_period.union(period)
        else:
            valid_periods.append(sorted(tmp_valid_period))
            tmp_valid_period = set(period)

    if tmp_valid_period:
        valid_periods.append(sorted(tmp_valid_period))

    contexts = []

    for valid_period in  valid_periods:
        period_context = df[df[YEAR_COL].isin(valid_period)]

        # Limit the number of decimals in the context.
        period_context = period_context[[YEAR_COL, VALUE_COL, IMPUTED_COL]].to_json(orient="records", double_precision=5)

        context = {
            "Indicator": indicator_name,
            "Country": country,
            "Series": json.loads(period_context)
        }

        contexts.append(context)

    return contexts

In [ ]:
# _df

In [ ]:
# c = extract_anomaly_contexts(_df, period_window=3, outlier_indicator_total=3)
# c

In [ ]:
# c = extract_anomaly_contexts(_df, period_window=3, outlier_indicator_total=3)
# c

# """
# [{'Indicator': 'Number of deaths ages 15-19 years',
#   'Country': 'Honduras',
#   'Series': [{'Year': 1995, 'Value': 916.0, 'Imputed': False},
#    {'Year': 1996, 'Value': 924.0, 'Imputed': False},
#    {'Year': 1997, 'Value': 930.0, 'Imputed': False},
#    {'Year': 1998, 'Value': 1693.0, 'Imputed': False},
#    {'Year': 1999, 'Value': 932.0, 'Imputed': False},
#    {'Year': 2000, 'Value': 931.0, 'Imputed': False},
#    {'Year': 2001, 'Value': 929.0, 'Imputed': False}]}]"""

In [ ]:
# {'Indicator': 'Number of deaths ages 15-19 years',
#   'Country': 'Honduras',
#   'Series': [{'Year': 1995, 'Value': 916.0, 'Imputed': False},
#    {'Year': 1996, 'Value': 924.0, 'Imputed': False},
#    {'Year': 1997, 'Value': 930.0, 'Imputed': False},
#    {'Year': 1998, 'Value': 1693.0, 'Imputed': False},
#    {'Year': 1999, 'Value': 932.0, 'Imputed': False},
#    {'Year': 2000, 'Value': 931.0, 'Imputed': False},
#    {'Year': 2001, 'Value': 929.0, 'Imputed': False}]}

## LLM Prompting

In [ ]:
# SYSTEM_PROMPT = """You are a data-quality and macroeconomic diagnostics assistant that validates anomalies in time series like the World Development Indicators (WDI).

# # GOAL:
# Identify and validate anomalies in indicator time series, then explain the most likely verifiable causes.

# # CONSTRAINTS:
# - Use only historically documented, verifiable facts (e.g., economic crises, conflicts, policy reforms, debt relief, rebasing events).
# - DO NOT speculate or hypothesize when facts are unclear.
# - If you cannot recall a well-documented event that explains the anomaly, classify it as "insufficient_data".
# - Provide only concise, factual explanations based on established macroeconomic or statistical history.
# - Every explanation must cite a known event or policy, with its approximate date range and type.
# - Never reveal internal reasoning; output only the structured JSON.
# - Only mark an anomaly as valid if there is at least one **well-documented** event or data-related cause.
# - Prefer “insufficient_data” over creative reasoning.
# - When unsure, lower confidence ≤ 0.5.
# - Always follow the JSON schema exactly.
# - If no concrete, verifiable source (event, policy, dataset, or document) can be named, return an empty "evidence_source": [].
# - Never invent or generalize sources (e.g., "IMF", "World Bank") just to populate fields. It is acceptable to leave evidence_source empty or use "verifiability": "not_applicable" for data errors or implausible magnitudes without a specific origin."""


SYSTEM_PROMPT = """You are a data-quality and macroeconomic diagnostics assistant that validates anomalies in time-series indicators such as those from the World Development Indicators (WDI).

# GOAL:
Identify and validate anomalies in indicator time series, explain the most likely verifiable causes, and classify each anomaly window.

# CONSTRAINTS:
- Use only historically documented, verifiable facts — including well-known global or national events that are publicly recognized and corroborated by multiple reputable sources (e.g., natural disasters, wars, global crises, major policy reforms).
- If you have access to the internet, always search the internet to find reasons, but only consider the result if it actually makes sense.
- You may use your general world knowledge to recall such events if they are clearly established in history.
- Do not speculate or hypothesize beyond documented or widely recognized events.
- If you cannot recall a clear, well-documented event that explains the anomaly, classify it as "insufficient_data".
- Provide concise, factual explanations; avoid storytelling or uncertain reasoning.
- Cite specific events or sources whenever possible with approximate date ranges and event types.
- Never reveal internal reasoning — output only the structured JSON that matches the schema.
- Only mark an anomaly as valid (is_anomaly=true) when there is at least one well-documented event or data-related cause.
- Prefer "insufficient_data" over creative inference when unsure.
- When in doubt, lower confidence ≤ 0.5.
- Use "YYYY-MM" format in `date_range`.
- Don't explain imputed values."""


USER_PROMPT_TEMPLATE = """# TASK
Validate the anomalies in the time series below, explain their most likely verifiable causes, and classify each anomaly window.

# ANALYSIS RULES
1. Treat anomalies as windows ([start, end]), not individual points; merge contiguous anomalous years.
2. Confirm anomalies only if they align with a verifiable event or clear data-quality issue.
3. The time series includes imputed values indicated by the "Imputed" column. Do not attempt to explain these values.
4. You may use general, well-documented historical knowledge (e.g., wars, natural disasters, global crises, pandemics, major policy reforms, or statistical revisions) when such events are clearly established in history and widely recognized.
5. If there is no match with known history or documented statistical events set is_anomaly=false and classification="insufficient_data".
6. Use one of these primary classifications:
   - "data_error" — placeholder, rounding, rebasing artifact, template issue, ingestion error, logical computation impossibility.
   - "external_driver" — macroeconomic or geopolitical event, conflict, policy reform, disaster, pandemic, global cycle.
   - "measurement_system_update" — rebasing, SNA/PPP revision, new census benchmark, classification change.
   - "modeling_artifact" — anomaly detector or transformation artifact.
   - "insufficient_data" — no verifiable cause.
7. Assign "evidence_strength" as one of:
   - "strong_direct" — clearly linked, well-documented event or revision.
   - "moderate_contextual" — plausible contextual relationship but without direct documentation.
   - "weak_speculative" — weak or uncertain linkage, minimal documentation.
   - "no_evidence" — used when no evidence_source applies (e.g., data error or missing data).
   Only "strong_direct" counts as a valid confirmation for is_anomaly=true.
8. Include an "evidence_source" list only when you can name a specific, verifiable event or document.
   - Use concise factual titles (e.g., "COVID-19 pandemic", "2014 GDP rebasing, Nigeria").
   - Do NOT invent or generalize sources (e.g., "IMF", "World Bank") to fill the field.
   - If no such source exists, use "evidence_source": [] and, if appropriate, set "verifiability": "not_applicable".
   - For generic data or reporting errors (e.g., implausible magnitudes, placeholders, ingestion artifacts), an empty evidence_source is expected and correct.
9. Keep explanations concise and factual, citing only verifiable events or mechanisms; avoid speculation or uncertain reasoning.
10. Output ONLY valid JSON matching the provided schema—no markdown, comments, or extra text.

# CONTEXT
{{INPUT_SERIES_INFO}}"""


ANOMALY_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
      "name": "anomaly_explanation",
      "strict": True,
      "schema": {
        "type": "object",
        "required": [
          "anomalies"
        ],
        "additionalProperties": False,
        "properties": {
          "anomalies": {
            "type": "array",
            "items": {
              "type": "object",
              "additionalProperties": False,
              "properties": {
                "window": {
                  "type": "array",
                  "items": {
                    "type": "integer"
                  },
                  "minItems": 2,
                  "maxItems": 2,
                  "description": "Inclusive [start_year, end_year] of the anomaly window."
                },
                "is_anomaly": {
                  "type": "boolean"
                },
                "classification": {
                  "type": "string",
                  "enum": [
                    "data_error",
                    "external_driver",
                    "measurement_system_update",
                    "modeling_artifact",
                    "insufficient_data"
                  ]
                },
                "confidence": {
                  "type": "number",
                  "minimum": 0,
                  "maximum": 1
                },
                "explanation": {
                  "type": "string"
                },
                "evidence_strength": {
                  "type": "string",
                  "enum": [
                    "strong_direct",
                    "moderate_contextual",
                    "weak_speculative",
                    "no_evidence"
                  ]
                },
                "evidence_source": {
                  "type": "array",
                  "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                      "name": {
                        "type": "string"
                      },
                      "date_range": {
                        "type": "string"
                      },
                      "source_type": {
                        "type": "string",
                        "enum": [
                          "global_event",
                          "regional_event",
                          "policy_change",
                          "data_revision",
                          "rebasing",
                          "natural_disaster",
                          "conflict",
                          "economic_crisis",
                          "data_error",
                          "other"
                        ]
                      },
                      "verifiability": {
                        "type": "string",
                        "enum": [
                          "well_documented",
                          "partially_documented",
                          "uncertain",
                          "not_applicable"
                        ]
                      }
                    },
                    "required": [
                      "name",
                      "date_range",
                      "source_type",
                      "verifiability"
                    ]
                  }
                },
                "source": {
                  "type": "string",
                  "enum": [
                    "llm_inferred"
                  ]
                }
              },
              "required": [
                "window",
                "is_anomaly",
                "classification",
                "confidence",
                "explanation",
                "evidence_strength",
                "evidence_source",
                "source"
              ]
            }
          }
        }
      }
    }
  }

# Render the template
user_prompt_template = Template(USER_PROMPT_TEMPLATE)

# input_context = c[0]
# filled_prompt = user_prompt_template.render(INPUT_SERIES_INFO=json.dumps(input_context, indent=2))

# print(filled_prompt)

In [ ]:
# json.dumps(ANOMALY_RESPONSE_FORMAT)

In [ ]:
# c

## LLM Inference

### Helper functions

In [ ]:
# openai_out_df["response"].iloc[0]

In [ ]:
from pydantic import ValidationError


def create_and_get_anomalies_csv(provider, batch_output_fname):
    print(batch_output_fname)

    assert provider in ["openai", "gemini"]
    assert os.path.exists(batch_output_fname)
    assert provider in batch_output_fname

    out_df = pd.read_json(batch_output_fname, lines=True)

    anomalies = []
    for ix, row in out_df.iterrows():
        if provider == "gemini":
            row["custom_id"] = row["key"]

        custom_id = row["custom_id"]
        _, _, indicator_code, country, _ = custom_id.split("-")

        if provider == "openai":
            try:
                content = json.loads(row["response"]["body"]["choices"][0]["message"]["content"])
            except KeyError:
                content = json.loads(row["response"]["body"]["output"][0]["content"][0]["text"])
        elif provider == "gemini":
            content = row["response"]["candidates"][0]["content"]["parts"][0]["text"]
            try:
                content = json.loads(
                    AnomalyExplanation.model_validate_json(
                        content
                    ).model_dump_json()
                )
            except ValidationError:
                print(f"Validation error for {custom_id}: {content}")
                continue
        else:
            raise ValueError(f"Unknown provider: {provider}")

        for anomaly in content["anomalies"]:
            anomaly = {
                "custom_id": custom_id,
                "indicator_code": indicator_code,
                "indicator": series_code_names[indicator_code],
                "country_code": country,
                "country": country_code_names[country],
                **anomaly
            }

            anomalies.append(anomaly)

    anomalies_fname = batch_output_fname.replace(".jsonl", ".csv")

    anomalies_df = pd.DataFrame(anomalies)
    anomalies_df["window_str"] = anomalies_df["window"].apply(lambda x: f"{x[0]}-{x[1]}")
    anomalies_df.to_csv(anomalies_fname)

    return anomalies_df

### OpenAI

#### Batch generation

In [ ]:
# from openai import OpenAI
# client = OpenAI()


def build_payload(endpoint, model_id: str, system_prompt: str, user_prompt: str, response_format: dict, with_search: bool = False, **api_kwargs):
    """Builds an API payload for anomaly explanation using an LLM.

    Args:
        input_context (dict): The context to be analyzed.
        endpoint (str): The endpoint for the LLM (completions / responses)
        model_id (str): The identifier for the LLM model to be used.
        with_search (bool):
        **api_kwargs: Additional keyword arguments to override default API parameters.

    Returns:
        dict: A payload dictionary containing:
            - messages: List of system and user messages with the prompt and content
            - response_format: JSON schema specification for the expected response
            - model: The specified model ID
            - Additional API parameters (temperature, max_completion_tokens, etc.)

    Notes:
        Default API parameters include:
        - temperature: 0
        - max_completion_tokens: 8192
        - top_p: 1
        - frequency_penalty: 0
        - presence_penalty: 0
        - seed: 1029
    """

    assert endpoint in ["completions", "responses"]

    default_kwargs = dict(
        model=model_id,
        temperature=0,
        max_completion_tokens=8192,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
        seed=1029,
        store=True
    )

    if api_kwargs:
        default_kwargs.update(api_kwargs)

    if with_search:
        if "tools" not in default_kwargs:
            default_kwargs["tools"] = [
                {
                "type": "web_search",
                "search_context_size": "low"
                }
            ]
        else:
            if "web_search" not in [tool["type"] for tool in default_kwargs["tools"]]:
                default_kwargs["tools"].append(
                    {
                    "type": "web_search",
                    "search_context_size": "low"
                    }
                )

        if "include" not in default_kwargs:
            default_kwargs["include"] = ["web_search_call.action.sources"]
        else:
            if "web_search_call.action.sources" not in default_kwargs["include"]:
                default_kwargs["include"].append("web_search_call.action.sources")

        # This will make force the LLM to always search
        default_kwargs["tool_choice"] = "required"

    messages = [
        {
          "role": "system",
          "content": [
            {
              "type": "text",
              "text": system_prompt
            }
          ]
        },
        {
          "role": "user",
          "content": [
            {
              "type": "text",
              "text": user_prompt
            }
          ]
        }
      ]

    if endpoint == "responses":
        for msg in messages:
            msg["content"][0]["type"] = "input_text"

        payload = dict(
            input=messages,
            text={
                "format": {"type": "json_schema", **response_format["json_schema"]}
            },
            **default_kwargs,
        )

        if "max_completion_tokens" in payload:
            payload["max_output_tokens"] = payload.pop("max_completion_tokens")

        payload.pop("frequency_penalty")
        payload.pop("presence_penalty")
        payload.pop("seed")

    else:
        payload = dict(
            messages=messages,
            response_format=response_format,
            **default_kwargs,
        )

    return payload

In [ ]:
import os
import pandas as pd
import json
from hashlib import md5
from datetime import datetime, timezone
from jinja2 import Template
from tqdm.auto import tqdm


model_id = "gpt-4.1-mini"
endpoint = "responses"
ENDPOINT_URL = {
    "completions": "/v1/chat/completions",
    "responses": "/v1/responses",
}
with_search = False

if with_search and endpoint == "completions":
    print("Search not supported in completions...")
    with_search = False

system_prompt = SYSTEM_PROMPT
pref = "search" if with_search else "nosearch"

prompt_hash = md5((system_prompt + "\n" + USER_PROMPT_TEMPLATE).encode()).hexdigest()[:8]
openai_payload_fname = os.path.join(LLM_INPUT_DIR, f"openai-{pref}-anomaly-detection-{int(datetime.now(tz=timezone.utc).timestamp())}.jsonl")

os.makedirs(os.path.dirname(openai_payload_fname), exist_ok=True)

rows = 0

with open(openai_payload_fname, "w") as fl:
    for (ix, row) in tqdm(list(shortlist.iterrows())):
        country = row[COUNTRY_CODE_COL]
        indicator_code = row[INDICATOR_CODE_COL]

        _df = source_df.loc[(indicator_code, country)].sort_values(YEAR_COL)
        _df = _df.reset_index()

        # _df = df[(df["Indicator.Code"] == indicator_code) & (df["Country.Code"] == country)].sort_values("Year")
        contexts = extract_anomaly_contexts(_df, period_window=3, outlier_indicator_total=3)

        for input_context in contexts:
            input_context = json.dumps(input_context, indent=2)

            user_prompt = user_prompt_template.render(INPUT_SERIES_INFO=input_context)

            payload = build_payload(endpoint=endpoint, model_id=model_id, system_prompt=system_prompt, user_prompt=user_prompt, response_format=ANOMALY_RESPONSE_FORMAT, with_search=with_search)

            batch_data = dict(
                custom_id=f"{pref}-{prompt_hash}-{indicator_code}-{country}-{md5(input_context.encode()).hexdigest()}",
                method="POST",
                url=ENDPOINT_URL[endpoint],
                body=payload,
            )
            fl.write(json.dumps(batch_data) + "\n")
            rows += 1
            # print(payload)

print(f"Wrote {rows} rows to {openai_payload_fname}")

  0%|          | 0/354 [00:00<?, ?it/s]

Wrote 354 rows to /content/anomaly-explanation/CSC_TOP_ANOMALIES_2026-02-06/llm-input/openai-nosearch-anomaly-detection-1770417716.jsonl


In [ ]:
contexts[-1]

{'Indicator': 'Percentage of population with access to electricity',
 'Country': 'Nauru',
 'Series': [{'YEAR': 2004, 'VALUE': 99.0, 'Imputed': False},
  {'YEAR': 2005, 'VALUE': 99.0, 'Imputed': False},
  {'YEAR': 2006, 'VALUE': 99.0, 'Imputed': False},
  {'YEAR': 2007, 'VALUE': 99.8, 'Imputed': False},
  {'YEAR': 2008, 'VALUE': 99.1, 'Imputed': False},
  {'YEAR': 2009, 'VALUE': 99.1, 'Imputed': False},
  {'YEAR': 2010, 'VALUE': 99.2, 'Imputed': False}]}

#### Run batch

In [ ]:
## Upload file to OpenAI
from google.colab import userdata
from openai import OpenAI

if with_search:
    raise ValueError("Web search is not supported in the Batch API")

openai_client = OpenAI(api_key=userdata.get('ANOMALY_OPENAI_API_KEY'))

openai_file = openai_client.files.create(
  file=open(openai_payload_fname, "rb"),
  purpose="batch",
)

## Create the batch
openai_batch = openai_client.batches.create(
  input_file_id=openai_file.id,
  endpoint=ENDPOINT_URL[endpoint],
  completion_window="24h"
)

openai_batch

Batch(id='batch_69866e8d92bc81909b1f87ae818f70c3', completion_window='24h', created_at=1770417805, endpoint='/v1/responses', input_file_id='file-Ha3WGH9MZiSHYJXm5wwj1B', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1770504205, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [ ]:
if with_search:
    raise ValueError("Web search is not supported in the Batch API")

tmp_openai_batch = openai_client.batches.retrieve(openai_batch.id)

if tmp_openai_batch.status == "completed":
    openai_batch_output_fname = openai_payload_fname.replace(".jsonl", "-output.jsonl")
    openai_batch_output_fname = openai_batch_output_fname.replace("/llm-input/", "/llm-output/")
    openai_batch_output_file = openai_client.files.content(tmp_openai_batch.output_file_id)

    os.makedirs(os.path.dirname(openai_batch_output_fname), exist_ok=True)

    print(f"Saving output: {openai_batch_output_fname}")
    with open(openai_batch_output_fname, "wb") as fout:
        fout.write(openai_batch_output_file.content)

tmp_openai_batch

Saving output: /content/anomaly-explanation/CSC_TOP_ANOMALIES_2026-02-06/llm-output/openai-nosearch-anomaly-detection-1770417716-output.jsonl


Batch(id='batch_69866e8d92bc81909b1f87ae818f70c3', completion_window='24h', created_at=1770417805, endpoint='/v1/responses', input_file_id='file-Ha3WGH9MZiSHYJXm5wwj1B', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1770417969, error_file_id=None, errors=None, expired_at=None, expires_at=1770504205, failed_at=None, finalizing_at=1770417952, in_progress_at=1770417867, metadata=None, model='gpt-4.1-mini-2025-04-14', output_file_id='file-Ae4PqwmWVtGnxxaqUbRzvd', request_counts=BatchRequestCounts(completed=354, failed=0, total=354), usage=BatchUsage(input_tokens=487610, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=35713, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=523323))

In [ ]:

# # country = "HND"
# # indicator_code = "SH.DTH.1519"


# # _df = source_df.loc[("DT.ODA.ODAT.XP.ZS", "COG")]
# _df = source_df.loc[("SH.DTH.1519", "HND")]
# _df = _df.reset_index()
# _df

# contexts = extract_anomaly_contexts(_df, period_window=3, outlier_indicator_total=3)
# input_context = contexts[0]

# input_context = json.dumps(input_context, indent=2)

# user_prompt = user_prompt_template.render(INPUT_SERIES_INFO=input_context)

# payload = build_payload(endpoint=endpoint, model_id=model_id, system_prompt=system_prompt, user_prompt=user_prompt, response_format=ANOMALY_RESPONSE_FORMAT, with_search=with_search)
# payload

In [ ]:
# payload

In [ ]:
# response = openai_client.responses.create(**payload)
# response

In [ ]:
# response.model_dump()

In [ ]:
# response.output

#### Analysis of Output

In [ ]:
# out_df = pd.read_json("/content/batch_68f59aac175881909097b64d72d590be_output.jsonl", lines=True)
openai_out_df = pd.read_json(openai_batch_output_fname, lines=True)

openai_out_df

,id,custom_id,response,error
0,batch_req_69866f213b588190a278e5c5808fb1b7,nosearch-c660ac92-WB_CSC_SI_POV_UMIC-ISL-f0071...,"{'status_code': 200, 'request_id': '634707ff-e...",NaN
1,batch_req_69866f21312081908ddf191a3cf2a668,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-MYS-5f6d7...,"{'status_code': 200, 'request_id': '1caaa7ef-b...",NaN
2,batch_req_69866f2139948190968c6ab24a3900cc,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-STP-bb9dc...,"{'status_code': 200, 'request_id': '353cb332-2...",NaN
3,batch_req_69866f212ba0819099d82850f945f331,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-MNE-6956c...,"{'status_code': 200, 'request_id': '89aafb59-9...",NaN
4,batch_req_69866f212fe081909a4372d6e5272a20,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-LKA-ae610...,"{'status_code': 200, 'request_id': '4e8c8f6d-d...",NaN
...,...,...,...,...
349,batch_req_69866f309d1481909fc1c2aa460b9ad1,nosearch-c660ac92-WB_CSC_SL_EMP_WORK_ZS-SYR-d5...,"{'status_code': 200, 'request_id': '523c8f6a-9...",NaN
350,batch_req_69866f30ab0c81908cfa321eba74231d,nosearch-c660ac92-WB_CSC_SH_H2O_BASW_ZS_CC-BFA...,"{'status_code': 200, 'request_id': 'bae8af15-d...",NaN
351,batch_req_69866f30ace88190b72ededcf5d39ae4,nosearch-c660ac92-WB_CSC_SH_H2O_BASW_ZS-BFA-40...,"{'status_code': 200, 'request_id': '31b9d29a-4...",NaN
352,batch_req_69866f30b7588190a847bd126d9a19d9,nosearch-c660ac92-WB_CSC_IT_NET_USER_ZS-TUV-fd...,"{'status_code': 200, 'request_id': '857606de-4...",NaN


In [ ]:
openai_anomalies_df = create_and_get_anomalies_csv(
    "openai",
    openai_batch_output_fname
)

openai_anomalies_df

/content/anomaly-explanation/CSC_TOP_ANOMALIES_2026-02-06/llm-output/openai-nosearch-anomaly-detection-1770417716-output.jsonl


,custom_id,indicator_code,indicator,country_code,country,window,is_anomaly,classification,confidence,explanation,evidence_strength,evidence_source,source,window_str
0,nosearch-c660ac92-WB_CSC_SI_POV_UMIC-ISL-f0071...,WB_CSC_SI_POV_UMIC,Percentage of global population living in pove...,ISL,Iceland,"[2011, 2011]",True,external_driver,0.9,The spike to 1.0% poverty in 2011 aligns with ...,strong_direct,"[{'name': 'Icelandic financial crisis impact',...",llm_inferred,2011-2011
1,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-MYS-5f6d7...,WB_CSC_EN_ATM_HAZA,Percentage of people exposed to hazardous air ...,MYS,Malaysia,"[2010, 2010]",True,external_driver,0.9,The spike in hazardous air quality exposure in...,strong_direct,"[{'name': '2010 Southeast Asian haze', 'date_r...",llm_inferred,2010-2010
2,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-STP-bb9dc...,WB_CSC_EN_ATM_HAZA,Percentage of people exposed to hazardous air ...,STP,Sao Tome and Principe,"[2016, 2016]",True,data_error,0.9,"The isolated increase to 0.2% in 2016, surroun...",no_evidence,[],llm_inferred,2016-2016
3,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-MNE-6956c...,WB_CSC_EN_ATM_HAZA,Percentage of people exposed to hazardous air ...,MNE,Montenegro,"[2012, 2012]",True,data_error,0.9,"The sudden increase to 1.7% in 2012, surrounde...",strong_direct,[{'name': 'Data inconsistency in Montenegro ai...,llm_inferred,2012-2012
4,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-LKA-ae610...,WB_CSC_EN_ATM_HAZA,Percentage of people exposed to hazardous air ...,LKA,Sri Lanka,"[2011, 2012]",True,measurement_system_update,0.9,The sudden increase in reported hazardous air ...,strong_direct,[{'name': 'Introduction of new air quality mon...,llm_inferred,2011-2012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266,nosearch-c660ac92-WB_CSC_NE_GDI_FPRV_ZS-SWZ-d9...,WB_CSC_NE_GDI_FPRV_ZS,Private investment as a percentage of GDP,SWZ,Eswatini,"[2020, 2020]",True,external_driver,0.9,Sharp drop in private investment in 2020 align...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
267,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-ARG-be...,WB_CSC_SM_POP_REFG_OR,Number of displaced people in need of protection,ARG,Argentina,"[2007, 2007]",True,external_driver,0.9,The sharp increase in displaced people in 2007...,strong_direct,"[{'name': '2007-2008 Global Financial Crisis',...",llm_inferred,2007-2007
268,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-ARG-be...,WB_CSC_SM_POP_REFG_OR,Number of displaced people in need of protection,ARG,Argentina,"[2008, 2010]",True,external_driver,0.9,The significant decrease in displaced people f...,strong_direct,[{'name': 'Argentina economic recovery post-20...,llm_inferred,2008-2010
269,nosearch-c660ac92-WB_CSC_SL_EMP_WORK_ZS-SYR-d5...,WB_CSC_SL_EMP_WORK_ZS,"Wage and salaried workers, total (% total empl...",SYR,Syrian Arab Republic,"[2007, 2008]",True,external_driver,0.8,The sharp increase in wage and salaried worker...,moderate_contextual,[{'name': 'Pre-civil war economic reforms in S...,llm_inferred,2007-2008


In [ ]:
openai_anomalies_df.groupby("classification")["confidence"].agg(count="count", confidence_mean="mean", confidence_std="std")

,count,confidence_mean,confidence_std
classification,,,
data_error,40,0.902500,0.027619
external_driver,201,0.881095,0.043483
insufficient_data,16,0.593750,0.208066
measurement_system_update,14,0.900000,0.000000


In [ ]:
# Check covid

openai_anomalies_df[
    (openai_anomalies_df["classification"] == "external_driver") &
    openai_anomalies_df["explanation"].str.lower().str.contains("covid")]

,custom_id,indicator_code,indicator,country_code,country,window,is_anomaly,classification,confidence,explanation,evidence_strength,evidence_source,source,window_str
13,nosearch-c660ac92-WB_CSC_EN_ATM_HAZA-KEN-d6e26...,WB_CSC_EN_ATM_HAZA,Percentage of people exposed to hazardous air ...,KEN,Kenya,"[2020, 2021]",True,external_driver,0.9,Increase in hazardous air quality exposure in ...,strong_direct,[{'name': 'COVID-19 pandemic and associated ai...,llm_inferred,2020-2021
15,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_FE_ME_ZS-...,WB_CSC_SL_UEM_NEET_FE_ME_ZS,"Youth not in education, employment, or trainin...",DOM,Dominican Republic,"[2020, 2020]",True,external_driver,0.9,"Sharp increase in youth not in education, empl...",strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
16,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_ME_ZS-VUT...,WB_CSC_SL_UEM_NEET_ME_ZS,"Youth not in education, employment, or trainin...",VUT,Vanuatu,"[2020, 2020]",True,external_driver,0.9,"The sharp increase in youth not in education, ...",strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
19,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_ME_ZS-DOM...,WB_CSC_SL_UEM_NEET_ME_ZS,"Youth not in education, employment, or trainin...",DOM,Dominican Republic,"[2020, 2020]",True,external_driver,0.9,"Sharp increase in youth not in education, empl...",strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
20,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_FE_ME_ZS-...,WB_CSC_SL_UEM_NEET_FE_ME_ZS,"Youth not in education, employment, or trainin...",VUT,Vanuatu,"[2020, 2020]",True,external_driver,0.9,The sharp increase in 2020 aligns with the COV...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_ME_ZS-DEA...,WB_CSC_SL_UEM_NEET_ME_ZS,"Youth not in education, employment, or trainin...",DEA,East Asia & Pacific (IDA total),"[2019, 2021]",True,external_driver,0.9,"Increase in youth not in education, employment...",strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2019-2021
253,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_ME_ZS-TGO...,WB_CSC_SL_UEM_NEET_ME_ZS,"Youth not in education, employment, or trainin...",TGO,Togo,"[2020, 2022]",True,external_driver,0.9,"Increase in youth not in education, employment...",strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2022
255,nosearch-c660ac92-WB_CSC_EN_ATM_GHGT_GT_CE-MEX...,WB_CSC_EN_ATM_GHGT_GT_CE,Total greenhouse gas emissions,MEX,Mexico,"[2019, 2020]",True,external_driver,0.9,Significant decrease in greenhouse gas emissio...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2019-2020
259,nosearch-c660ac92-WB_CSC_SL_EMP_WORK_FE_ZS-MEX...,WB_CSC_SL_EMP_WORK_FE_ZS,"Wage and salaried workers, female (% female em...",MEX,Mexico,"[2020, 2020]",True,external_driver,0.9,The sharp increase in female wage and salaried...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020


### Gemini

#### Batch Generation

In [ ]:
## Define the pydantic models for the schema
from enum import Enum
from typing import List
from typing_extensions import Annotated

from pydantic import BaseModel, Field, ConfigDict


# ----- Enums ----- #

class Classification(str, Enum):
    data_error = "data_error"
    external_driver = "external_driver"
    measurement_system_update = "measurement_system_update"
    modeling_artifact = "modeling_artifact"
    insufficient_data = "insufficient_data"


class EvidenceStrength(str, Enum):
    strong_direct = "strong_direct"
    moderate_contextual = "moderate_contextual"
    weak_speculative = "weak_speculative"
    no_evidence = "no_evidence"


class EvidenceSourceType(str, Enum):
    global_event = "global_event"
    regional_event = "regional_event"
    policy_change = "policy_change"
    data_revision = "data_revision"
    rebasing = "rebasing"
    natural_disaster = "natural_disaster"
    conflict = "conflict"
    economic_crisis = "economic_crisis"
    data_error = "data_error"
    other = "other"


class Verifiability(str, Enum):
    well_documented = "well_documented"
    partially_documented = "partially_documented"
    uncertain = "uncertain"
    not_applicable = "not_applicable"


class Source(str, Enum):
    llm_inferred = "llm_inferred"


# ----- Strict Base ----- #

class StrictBaseModel(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        strict=True,
    )


# ----- Nested models ----- #

class EvidenceSource(StrictBaseModel):
    name: str
    date_range: str
    source_type: EvidenceSourceType
    verifiability: Verifiability


class Anomaly(StrictBaseModel):
    window: Annotated[
        List[int],
        Field(min_length=2, max_length=2, description="Inclusive [start_year, end_year] of the anomaly window.")
    ]

    is_anomaly: bool
    classification: Classification

    confidence: Annotated[
        float,
        Field(ge=0, le=1)
    ]

    explanation: str
    evidence_strength: EvidenceStrength

    evidence_source: List[EvidenceSource]

    source: Source


class AnomalyExplanation(StrictBaseModel):
    anomalies: List[Anomaly]


# AnomalyExplanation.model_json_schema()

In [ ]:
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=userdata.get('ANOMALY_GEMINI_API_KEY'))
gemini_payload_fname = openai_payload_fname.replace("/openai-", "/gemini-")
print(gemini_payload_fname)
os.makedirs(os.path.dirname(gemini_payload_fname), exist_ok=True)

with open(gemini_payload_fname, "w") as fout:
    with open(openai_payload_fname) as fin:
        for line in fin:
            exp_payload = json.loads(line.strip())

            if exp_payload["url"] == "/v1/responses":
                system_msg, user_msg = exp_payload["body"]["input"]
            else:
                system_msg, user_msg = exp_payload["body"]["messages"]

            line = dict(
                key=exp_payload["custom_id"],
                request=dict(
                    contents=[
                        dict(
                            parts=[dict(
                                text=user_msg["content"][0]["text"]
                            )]
                        )
                    ],
                    systemInstruction=dict(
                        parts=[dict(
                            text=system_msg["content"][0]["text"]
                        )]
                    ),
                    generationConfig=dict(
                        responseMimeType="application/json",
                        responseJsonSchema=AnomalyExplanation.model_json_schema(),
                        temperature=exp_payload["body"].get("temperature", 1),
                        topP=exp_payload["body"].get("top_p", 1),
                        maxOutputTokens=exp_payload["body"].get("max_output_tokens", 8192),
                        seed=exp_payload["body"].get("seed", 1029),
                        frequencyPenalty=exp_payload["body"].get("frequency_penalty", 0),
                        presencePenalty=exp_payload["body"].get("presence_penalty", 0),
                    )
                )
            )

            fout.write(json.dumps(line))
            fout.write('\n')


# Upload the file to the File API
uploaded_file = gemini_client.files.upload(
    file=gemini_payload_fname,
    config=types.UploadFileConfig(display_name=gemini_payload_fname, mime_type='jsonl')
)

print(f"Uploaded file: {uploaded_file.name}")

/content/anomaly-explanation/CSC_TOP_ANOMALIES_2026-02-06/llm-input/gemini-nosearch-anomaly-detection-1770417716.jsonl
Uploaded file: files/ekh3gs70otuu


In [ ]:
gemini_payload_fname

'/content/anomaly-explanation/CSC_TOP_ANOMALIES_2026-02-06/llm-input/gemini-nosearch-anomaly-detection-1770417716.jsonl'

#### Run batch

In [ ]:
gemini_file_batch_job = gemini_client.batches.create(
    model="gemini-2.5-flash",
    src=uploaded_file.name,
    config={
        'display_name': f"{gemini_payload_fname}-job-1",
    },
)

print(f"Created batch job: {gemini_file_batch_job.name}")

Created batch job: batches/b1rheav53xo6psj8tcb7g1hes9n57vug6h68


In [ ]:
batch_job = gemini_client.batches.get(name=gemini_file_batch_job.name)

gemini_output_fname = gemini_payload_fname.replace(".jsonl", "-output.jsonl")
gemini_output_fname = gemini_output_fname.replace("/llm-input/", "/llm-output/")
os.makedirs(os.path.dirname(gemini_output_fname), exist_ok=True)


if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

    # If batch job was created with a file
    if batch_job.dest and batch_job.dest.file_name:
        # Results are in a file
        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("Downloading result file content...")
        file_content = gemini_client.files.download(file=result_file_name)
        # Process file_content (bytes) as needed

        with open(gemini_output_fname, "w") as fout:
            fout.write(file_content.decode('utf-8'))

    # If batch job was created with inline request
    # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
    elif batch_job.dest and batch_job.dest.inlined_responses:
        # Results are inline
        print("Results are inline:")
        for i, inline_response in enumerate(batch_job.dest.inlined_responses):
            print(f"Response {i+1}:")
            if inline_response.response:
                # Accessing response, structure may vary.
                try:
                    print(inline_response.response.text)
                except AttributeError:
                    print(inline_response.response) # Fallback
            elif inline_response.error:
                print(f"Error: {inline_response.error}")
    else:
        print("No results found (neither file nor inline).")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")
    if batch_job.error:
        print(f"Error: {batch_job.error}")

Results are in file: files/batch-b1rheav53xo6psj8tcb7g1hes9n57vug6h68


In [ ]:
gemini_output_df = pd.read_json(gemini_output_fname, lines=True)
gemini_output_df

,response,key
0,"{'responseId': 'vXWGaaL6GtvAqtsP47W40Q4', 'can...",nosearch-c660ac92-WB_CSC_SH_STA_BASS_ZS_CC-COM...
1,"{'candidates': [{'content': {'role': 'model', ...",nosearch-c660ac92-WB_CSC_SH_STA_BASS_ZS-COM-71...
2,"{'responseId': 'yHWGacWuCdboz7IPn9X74A8', 'usa...",nosearch-c660ac92-WB_CSC_NE_GDI_FPRV_ZS-GMB-fb...
3,"{'modelVersion': 'gemini-2.5-flash', 'candidat...",nosearch-c660ac92-WB_CSC_SL_UEM_NEET_ME_ZS-GUY...
4,"{'modelVersion': 'gemini-2.5-flash', 'response...",nosearch-c660ac92-WB_CSC_SL_UEM_NEET_FE_ME_ZS-...
...,...,...
349,"{'modelVersion': 'gemini-2.5-flash', 'candidat...",nosearch-c660ac92-WB_CSC_SL_EMP_WORK_ZS-SYR-d5...
350,"{'usageMetadata': {'totalTokenCount': 5421, 'c...",nosearch-c660ac92-WB_CSC_SH_H2O_BASW_ZS_CC-BFA...
351,"{'usageMetadata': {'candidatesTokenCount': 11,...",nosearch-c660ac92-WB_CSC_SH_H2O_BASW_ZS-BFA-40...
352,"{'candidates': [{'content': {'role': 'model', ...",nosearch-c660ac92-WB_CSC_IT_NET_USER_ZS-TUV-fd...


In [ ]:
gemini_anomalies_df = create_and_get_anomalies_csv(
    "gemini",
    gemini_output_fname
)

gemini_anomalies_df

/content/anomaly-explanation/CSC_TOP_ANOMALIES_2026-02-06/llm-output/gemini-nosearch-anomaly-detection-1770417716-output.jsonl


,custom_id,indicator_code,indicator,country_code,country,window,is_anomaly,classification,confidence,explanation,evidence_strength,evidence_source,source,window_str
0,nosearch-c660ac92-WB_CSC_SH_STA_BASS_ZS-COM-71...,WB_CSC_SH_STA_BASS_ZS,Percentage of people with access to basic sani...,COM,Comoros,"[1998, 1999]",False,insufficient_data,1.0,Data for these years is missing and not impute...,no_evidence,[],llm_inferred,1998-1999
1,nosearch-c660ac92-WB_CSC_NE_GDI_FPRV_ZS-GMB-fb...,WB_CSC_NE_GDI_FPRV_ZS,Private investment as a percentage of GDP,GMB,"Gambia, The","[2020, 2021]",True,data_error,0.8,The significant increase in private investment...,strong_direct,[{'name': 'IMF Country Report No. 20/290 on Th...,llm_inferred,2020-2021
2,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_ME_ZS-GUY...,WB_CSC_SL_UEM_NEET_ME_ZS,"Youth not in education, employment, or trainin...",GUY,Guyana,"[2019, 2020]",True,external_driver,0.9,"The increase in youth not in education, employ...",strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2019-2020
3,nosearch-c660ac92-WB_CSC_SL_UEM_NEET_FE_ME_ZS-...,WB_CSC_SL_UEM_NEET_FE_ME_ZS,"Youth not in education, employment, or trainin...",BOL,Bolivia,"[2015, 2016]",False,insufficient_data,0.5,"No specific, well-documented event or data iss...",no_evidence,[],llm_inferred,2015-2016
4,nosearch-c660ac92-WB_CSC_SL_EMP_WORK_ZS-MDG-d1...,WB_CSC_SL_EMP_WORK_ZS,"Wage and salaried workers, total (% total empl...",MDG,Madagascar,"[2002, 2003]",True,external_driver,1.0,The significant decline in wage and salaried w...,strong_direct,"[{'name': '2002 Malagasy political crisis', 'd...",llm_inferred,2002-2003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,nosearch-c660ac92-WB_CSC_SL_EMP_WORK_FE_ZS-LAO...,WB_CSC_SL_EMP_WORK_FE_ZS,"Wage and salaried workers, female (% female em...",LAO,Lao PDR,"[2020, 2020]",True,external_driver,0.9,The slight decrease in the percentage of femal...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
362,nosearch-c660ac92-WB_CSC_SL_EMP_WORK_ZS-SYR-d5...,WB_CSC_SL_EMP_WORK_ZS,"Wage and salaried workers, total (% total empl...",SYR,Syrian Arab Republic,"[2008, 2010]",True,measurement_system_update,0.9,The significant increase in the percentage of ...,strong_direct,"[{'name': 'Syria Labor Force Survey 2008', 'da...",llm_inferred,2008-2010
363,nosearch-c660ac92-WB_CSC_SH_H2O_BASW_ZS_CC-BFA...,WB_CSC_SH_H2O_BASW_ZS_CC,People using at least basic drinking water ser...,BFA,Burkina Faso,"[2017, 2022]",False,insufficient_data,0.5,The observed minor fluctuations (a slight decl...,no_evidence,[],llm_inferred,2017-2022
364,nosearch-c660ac92-WB_CSC_IT_NET_USER_ZS-TUV-fd...,WB_CSC_IT_NET_USER_ZS,Percentage of population using the internet,TUV,Tuvalu,"[2018, 2024]",False,insufficient_data,0.8,The time series contains only one non-null dat...,no_evidence,[],llm_inferred,2018-2024


### Analysis

In [ ]:
overlap_anomalies_df = openai_anomalies_df.merge(gemini_anomalies_df, on=["custom_id", "indicator_code", "indicator", "country_code", "country", "window_str"], how="outer", suffixes=("_openai", "_gemini"))
overlap_anomalies_df

,custom_id,indicator_code,indicator,country_code,country,window_openai,is_anomaly_openai,classification_openai,confidence_openai,explanation_openai,...,source_openai,window_str,window_gemini,is_anomaly_gemini,classification_gemini,confidence_gemini,explanation_gemini,evidence_strength_gemini,evidence_source_gemini,source_gemini
0,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-ALB-ed...,WB_CSC_EG_ELC_ACCS_ZS,Percentage of population with access to electr...,ALB,Albania,NaN,NaN,NaN,NaN,NaN,...,NaN,2008-2009,"[2008, 2009]",False,insufficient_data,0.5,While the sudden increase to exactly 100% in 2...,no_evidence,[],llm_inferred
1,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-ATG-c9...,WB_CSC_EG_ELC_ACCS_ZS,Percentage of population with access to electr...,ATG,Antigua and Barbuda,"[2003, 2005]",True,external_driver,0.9,The drop in electricity access from 100% in 20...,...,llm_inferred,2003-2005,"[2003, 2005]",False,insufficient_data,0.4,No well-documented external event or data revi...,no_evidence,[],llm_inferred
2,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-AZE-78...,WB_CSC_EG_ELC_ACCS_ZS,Percentage of population with access to electr...,AZE,Azerbaijan,NaN,NaN,NaN,NaN,NaN,...,NaN,2002-2002,"[2002, 2002]",False,insufficient_data,0.4,"No verifiable historical event, policy change,...",no_evidence,[],llm_inferred
3,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-BDI-ab...,WB_CSC_EG_ELC_ACCS_ZS,Percentage of population with access to electr...,BDI,Burundi,"[2006, 2007]",True,external_driver,0.8,The sharp drop in access to electricity in 200...,...,llm_inferred,2006-2007,"[2006, 2007]",True,external_driver,0.9,The significant increase in electricity access...,strong_direct,[{'name': 'End of Burundian Civil War and post...,llm_inferred
4,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-BEC-85...,WB_CSC_EG_ELC_ACCS_ZS,Percentage of population with access to electr...,BEC,Europe & Central Asia (IBRD only),"[2015, 2015]",True,external_driver,0.8,The drop in electricity access in 2015 in Euro...,...,llm_inferred,2015-2015,"[2015, 2015]",False,insufficient_data,0.5,A temporary dip in the percentage of populatio...,no_evidence,[],llm_inferred
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-TGO-c0...,WB_CSC_SM_POP_REFG_OR,Number of displaced people in need of protection,TGO,Togo,NaN,NaN,NaN,NaN,NaN,...,NaN,2005-2008,"[2005, 2008]",True,external_driver,1.0,The sharp increase in displaced people in Togo...,strong_direct,[{'name': 'Togo 2005 Presidential Election Cri...,llm_inferred
534,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-THA-bc...,WB_CSC_SM_POP_REFG_OR,Number of displaced people in need of protection,THA,Thailand,"[2006, 2008]",True,external_driver,0.9,Significant increase in displaced people in Th...,...,llm_inferred,2006-2008,"[2006, 2008]",True,external_driver,0.9,The significant increase in the number of disp...,strong_direct,[{'name': 'Escalation of Southern Thailand Ins...,llm_inferred
535,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-TKM-3e...,WB_CSC_SM_POP_REFG_OR,Number of displaced people in need of protection,TKM,Turkmenistan,"[2022, 2023]",True,external_driver,0.9,Significant increase in displaced people in Tu...,...,llm_inferred,2022-2023,"[2022, 2023]",True,measurement_system_update,1.0,The significant increase in the number of disp...,strong_direct,[{'name': 'UNHCR Global Trends 2022 Report: Co...,llm_inferred
536,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-TLA-df...,WB_CSC_SM_POP_REFG_OR,Number of displaced people in need of protection,TLA,Latin America & Caribbean (IDA & IBRD),"[2006, 2007]",True,external_driver,0.9,Sharp increase in displaced people in 2007 ali...,...,llm_inferred,2006-2007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
overlap_anomalies_df.to_csv(os.path.join(PROCESSED_DIR, "combined_explanations.csv"))

In [ ]:
overlap_explanations = overlap_anomalies_df[[
    "custom_id",
    "indicator",
    "country_code",
    "window_str",
    "explanation_openai", "explanation_gemini",
    "classification_openai", "classification_gemini",
    "evidence_source_openai", "evidence_source_gemini",
]].copy()
overlap_explanations.dropna(axis=0, thresh=10, inplace=True)

overlap_explanations.to_csv(os.path.join(PROCESSED_DIR, "overlap_explanations.csv"))
overlap_explanations

,custom_id,indicator,country_code,window_str,explanation_openai,explanation_gemini,classification_openai,classification_gemini,evidence_source_openai,evidence_source_gemini
1,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-ATG-c9...,Percentage of population with access to electr...,ATG,2003-2005,The drop in electricity access from 100% in 20...,No well-documented external event or data revi...,external_driver,insufficient_data,[{'name': 'Hurricane Ivan impact on Antigua an...,[]
3,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-BDI-ab...,Percentage of population with access to electr...,BDI,2006-2007,The sharp drop in access to electricity in 200...,The significant increase in electricity access...,external_driver,external_driver,[{'name': 'Burundi Civil War and Post-Conflict...,[{'name': 'End of Burundian Civil War and post...
4,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-BEC-85...,Percentage of population with access to electr...,BEC,2015-2015,The drop in electricity access in 2015 in Euro...,A temporary dip in the percentage of populatio...,external_driver,insufficient_data,[{'name': 'Ukraine crisis and conflict impact ...,[]
33,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-KHM-ae...,Percentage of population with access to electr...,KHM,2017-2018,The sharp increase in electricity access in 20...,The sharp increase in electricity access in 20...,measurement_system_update,insufficient_data,[{'name': '2017 Cambodia electricity access da...,[]
34,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-LBN-e9...,Percentage of population with access to electr...,LBN,2007-2007,The drop in electricity access in 2007 aligns ...,The 2006 Lebanon War caused significant damage...,external_driver,external_driver,"[{'name': '2006 Lebanon War', 'date_range': '2...","[{'name': '2006 Lebanon War', 'date_range': '2..."
...,...,...,...,...,...,...,...,...,...,...
524,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-PNG-e3...,Number of displaced people in need of protection,PNG,2022-2023,Significant increase in displaced people in 20...,The significant increase in displaced people i...,external_driver,external_driver,[{'name': '2022 Papua New Guinea volcanic erup...,"[{'name': '2022 Papua New Guinea earthquake', ..."
527,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-RUS-e1...,Number of displaced people in need of protection,RUS,2006-2006,The spike in displaced people in 2006 aligns w...,"No clear, well-documented event or data-relate...",external_driver,insufficient_data,[{'name': 'Second Chechen War aftermath and No...,[]
528,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-SLE-01...,Number of displaced people in need of protection,SLE,2000-2003,The sharp decline in displaced people from 200...,The high number of displaced people in 2000 wa...,external_driver,external_driver,"[{'name': 'Sierra Leone Civil War', 'date_rang...",[{'name': 'Sierra Leone Civil War and Post-Con...
534,nosearch-c660ac92-WB_CSC_SM_POP_REFG_OR-THA-bc...,Number of displaced people in need of protection,THA,2006-2008,Significant increase in displaced people in Th...,The significant increase in the number of disp...,external_driver,external_driver,[{'name': '2006-2008 Myanmar internal conflict...,[{'name': 'Escalation of Southern Thailand Ins...


In [ ]:
# !zip -r WDI_POV_ANOMALIES_2025-12-20.zip /content/WDI_POV_ANOMALIES_2025-12-20.CSV

  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/ (stored 0%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/processed/ (stored 0%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/processed/overlap_explanations.csv (deflated 81%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/processed/combined_explanations.csv (deflated 82%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/llm-input/ (stored 0%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/llm-input/gemini-nosearch-anomaly-detection-1766277638.jsonl (deflated 98%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/llm-input/openai-nosearch-anomaly-detection-1766277638.jsonl (deflated 98%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/.ipynb_checkpoints/ (stored 0%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/llm-output/ (stored 0%)
  adding: content/WDI_POV_ANOMALIES_2025-12-20.CSV/llm-output/gemini-nosearch-anomaly-detection-1766277638-output.csv (deflated 80%)
  adding: content/WDI_POV_ANOMALIES_2025

In [ ]:
!cd /content/anomaly-explanation/ && zip -r CSC_TOP_ANOMALIES_2026-02-06.zip CSC_TOP_ANOMALIES_2026-02-06

  adding: CSC_TOP_ANOMALIES_2026-02-06/ (stored 0%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-input/ (stored 0%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-input/.ipynb_checkpoints/ (stored 0%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-input/openai-nosearch-anomaly-detection-1770417716.jsonl (deflated 98%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-input/gemini-nosearch-anomaly-detection-1770417716.jsonl (deflated 98%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-output/ (stored 0%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-output/openai-nosearch-anomaly-detection-1770417716-output.jsonl (deflated 93%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-output/gemini-nosearch-anomaly-detection-1770417716-output.jsonl (deflated 86%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-output/openai-nosearch-anomaly-detection-1770417716-output.csv (deflated 81%)
  adding: CSC_TOP_ANOMALIES_2026-02-06/llm-output/gemini-nosearch-anomaly-detection-1770417716-output.csv (deflated 80%)
  adding: CSC_TOP_ANOMAL

In [ ]:
overlap_explanations.iloc[9].to_dict()

{'custom_id': 'nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-TGO-29aa5033af3f28c5e683e91a1f09e8ec',
 'indicator': 'Percentage of population with access to electricity',
 'country_code': 'TGO',
 'window_str': '2010-2011',
 'explanation_openai': 'The sharp drop in access to electricity in 2010 followed by a large increase in 2011 aligns with documented power supply disruptions and subsequent recovery efforts in Togo during this period, including infrastructure challenges and regional power shortages in West Africa.',
 'explanation_gemini': 'No clear, well-documented event or data revision found to explain the sharp decrease in electricity access in 2010 followed by a significant increase in 2011.',
 'classification_openai': 'external_driver',
 'classification_gemini': 'insufficient_data',
 'evidence_source_openai': [{'name': 'West African Power Supply Crisis',
   'date_range': '2009-2011',
   'source_type': 'regional_event',
   'verifiability': 'well_documented'}],
 'evidence_source_gemini': [

In [ ]:
openai_inp_df = pd.read_json("/content/llm-input/openai-anomaly-detection-1764880967.jsonl", lines=True)
openai_inp_df.head()

,custom_id,method,url,body
0,11c30231-NY.GDP.DISC.KN-SEN-3224f6e23f066db727...,POST,/v1/chat/completions,"{'messages': [{'role': 'system', 'content': [{..."
1,11c30231-IQ.CPA.FISP.XQ-KHM-05980f2833b54b6992...,POST,/v1/chat/completions,"{'messages': [{'role': 'system', 'content': [{..."
2,11c30231-NY.TTF.GNFS.KN-TLS-46e234b78ebaa5678d...,POST,/v1/chat/completions,"{'messages': [{'role': 'system', 'content': [{..."
3,11c30231-GC.TAX.OTHR.RV.ZS-CHE-fdc5a5034340b78...,POST,/v1/chat/completions,"{'messages': [{'role': 'system', 'content': [{..."
4,11c30231-SL.FAM.WORK.ZS-ZMB-dae230e6a7bbe5598a...,POST,/v1/chat/completions,"{'messages': [{'role': 'system', 'content': [{..."


In [ ]:
system_msg, user_msg = openai_inp_df["body"][0]["messages"]

In [ ]:
system_msg["content"][0]["text"]

'You are a data-quality and macroeconomic diagnostics assistant that validates anomalies in time-series indicators such as those from the World Development Indicators (WDI).\n\n# GOAL:\nIdentify and validate anomalies in indicator time series, explain the most likely verifiable causes, and classify each anomaly window.\n\n# CONSTRAINTS:\n- Use only historically documented, verifiable facts — including well-known global or national events that are publicly recognized and corroborated by multiple reputable sources (e.g., natural disasters, wars, global crises, major policy reforms).\n- You may use your general world knowledge to recall such events if they are clearly established in history.\n- Do not speculate or hypothesize beyond documented or widely recognized events.\n- If you cannot recall a clear, well-documented event that explains the anomaly, classify it as "insufficient_data".\n- Provide concise, factual explanations; avoid storytelling or uncertain reasoning.\n- Cite specific 

In [ ]:
user_msg["content"][0]["text"]

'# TASK\nValidate the anomalies in the time series below, explain their most likely verifiable causes, and classify each anomaly window.\n\n# ANALYSIS RULES\n1. Treat anomalies as windows ([start, end]), not individual points; merge contiguous anomalous years.\n2. Confirm anomalies only if they align with a verifiable event or clear data-quality issue.\n3. The time series includes imputed values indicated by the "Imputed" column. Do not attempt to explain these values.\n4. You may use general, well-documented historical knowledge (e.g., wars, natural disasters, global crises, pandemics, major policy reforms, or statistical revisions) when such events are clearly established in history and widely recognized.\n5. If there is no match with known history or documented statistical events set is_anomaly=false and classification="insufficient_data".\n6. Use one of these primary classifications:\n   - "data_error" — placeholder, rounding, rebasing artifact, template issue, ingestion error, log

### 1. Displaying the DataFrame to re-familiarize with its structure

In [ ]:
display(overlap_explanations.head())

,custom_id,indicator,country_code,window_str,explanation_openai,explanation_gemini,classification_openai,classification_gemini,evidence_source_openai,evidence_source_gemini
1,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-ATG-c9...,Percentage of population with access to electr...,ATG,2003-2005,The drop in electricity access from 100% in 20...,No well-documented external event or data revi...,external_driver,insufficient_data,[{'name': 'Hurricane Ivan impact on Antigua an...,[]
3,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-BDI-ab...,Percentage of population with access to electr...,BDI,2006-2007,The sharp drop in access to electricity in 200...,The significant increase in electricity access...,external_driver,external_driver,[{'name': 'Burundi Civil War and Post-Conflict...,[{'name': 'End of Burundian Civil War and post...
4,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-BEC-85...,Percentage of population with access to electr...,BEC,2015-2015,The drop in electricity access in 2015 in Euro...,A temporary dip in the percentage of populatio...,external_driver,insufficient_data,[{'name': 'Ukraine crisis and conflict impact ...,[]
33,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-KHM-ae...,Percentage of population with access to electr...,KHM,2017-2018,The sharp increase in electricity access in 20...,The sharp increase in electricity access in 20...,measurement_system_update,insufficient_data,[{'name': '2017 Cambodia electricity access da...,[]
34,nosearch-c660ac92-WB_CSC_EG_ELC_ACCS_ZS-LBN-e9...,Percentage of population with access to electr...,LBN,2007-2007,The drop in electricity access in 2007 aligns ...,The 2006 Lebanon War caused significant damage...,external_driver,external_driver,"[{'name': '2006 Lebanon War', 'date_range': '2...","[{'name': '2006 Lebanon War', 'date_range': '2..."


In [ ]:
# The 'is_anomaly_openai' and 'is_anomaly_gemini' columns are not present in `overlap_explanations`.
# Therefore, direct comparison of 'is_anomaly' agreement cannot be performed with the current DataFrame.

In [ ]:
# The 'confidence_openai' and 'confidence_gemini' columns are not present in `overlap_explanations`.
# Therefore, direct comparison of confidence scores cannot be performed with the current DataFrame.

### 2. Agreement on `is_anomaly` (if both columns exist and are not NaN)

In [ ]:
import numpy as np

# Filter out rows where either model didn't provide an 'is_anomaly' value
compare_anomaly_df = overlap_explanations.dropna(subset=['is_anomaly_openai', 'is_anomaly_gemini'])

# Calculate agreement
agreement = (compare_anomaly_df['is_anomaly_openai'] == compare_anomaly_df['is_anomaly_gemini']).mean() * 100
disagreement = 100 - agreement

print(f"Models agree on 'is_anomaly' for {agreement:.2f}% of the common entries.")
print(f"Models disagree on 'is_anomaly' for {disagreement:.2f}% of the common entries.")

# Display disagreements if any
disagreement_cases = compare_anomaly_df[compare_anomaly_df['is_anomaly_openai'] != compare_anomaly_df['is_anomaly_gemini']]
if not disagreement_cases.empty:
    print("\nCases where 'is_anomaly' differs:")
    display(disagreement_cases[['indicator', 'country_code', 'window_str', 'is_anomaly_openai', 'is_anomaly_gemini']])
else:
    print("\nNo disagreements found for 'is_anomaly'.")

KeyError: ['is_anomaly_openai', 'is_anomaly_gemini']

In [ ]:
# Filter out rows where either model didn't provide a classification
compare_classification_df = overlap_explanations.dropna(subset=['classification_openai', 'classification_gemini'])

classification_crosstab = pd.crosstab(
    compare_classification_df['classification_openai'],
    compare_classification_df['classification_gemini'],
    margins=True, # Add row/column sums
    dropna=False
)

print("Cross-tabulation of Anomaly Classifications:")
display(classification_crosstab)

### 3. Cross-tabulation of Classifications

In [ ]:
# Filter out rows where either model didn't provide a classification
compare_classification_df = overlap_explanations.dropna(subset=['classification_openai', 'classification_gemini'])

classification_crosstab = pd.crosstab(
    compare_classification_df['classification_openai'],
    compare_classification_df['classification_gemini'],
    margins=True, # Add row/column sums
    dropna=False
)

print("Cross-tabulation of Anomaly Classifications:")
display(classification_crosstab)

Cross-tabulation of Anomaly Classifications:


classification_gemini,data_error,external_driver,insufficient_data,measurement_system_update,All
classification_openai,,,,,
data_error,2,2,10,1,15
external_driver,3,53,16,2,74
insufficient_data,0,0,4,0,4
measurement_system_update,0,1,5,0,6
All,5,56,35,3,99


### 4. Comparing Confidence Scores

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter out rows where either model didn't provide a confidence score
compare_confidence_df = overlap_explanations.dropna(subset=['confidence_openai', 'confidence_gemini'])

plt.figure(figsize=(10, 6))
sns.scatterplot(x='confidence_openai', y='confidence_gemini', data=compare_confidence_df)
plt.title('OpenAI vs. Gemini Confidence Scores')
plt.xlabel('OpenAI Confidence')
plt.ylabel('Gemini Confidence')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Perfect Agreement') # Diagonal line for perfect agreement
plt.legend()
plt.grid(True)
plt.show()

print("\nMean Confidence Scores:")
print(f"OpenAI Mean Confidence: {compare_confidence_df['confidence_openai'].mean():.2f}")
print(f"Gemini Mean Confidence: {compare_confidence_df['confidence_gemini'].mean():.2f}")

print("\nCorrelation between Confidence Scores:")
print(compare_confidence_df[['confidence_openai', 'confidence_gemini']].corr())

KeyError: ['confidence_openai', 'confidence_gemini']

# Scratch

In [ ]:
df = pd.read_csv("top 30 anomalies.csv", index_col=0)

In [ ]:
def extract_anomaly_context(df, ix):
    pass

In [ ]:
df["outlier years"]

,outlier years
1,2006
2,2006
3,2005
4,1978
5,1985
6,2021
7,2022
8,1966 2022 2023
9,2022
10,2022


In [ ]:
period_window = 3
ix = 1
periods = df["outlier years"].loc[ix].split()

for period in periods:
    period = int(period)
    period_context = list(map(str, list(range(period - period_window, period + period_window + 1))))
    period_context = [i for i in df.columns if i in period_context]

In [ ]:
df.loc[ix, period_context]

,1
2003,-151890000.0
2004,-149239000.0
2005,-125313000.0
2006,-661020000.0
2007,-5885000.0
2008,-102165000.0
2009,-506000.0


In [ ]:
df.loc[ix, df.columns[:4].to_list() + period_context]

,1
Country.Name,Algeria
Country.Code,DZA
Indicator.Code,DT.NFL.MIBR.CD
Indicator.Name,"Net financial flows, IBRD (NFL, current US$)"
2003,-151890000.0
2004,-149239000.0
2005,-125313000.0
2006,-661020000.0
2007,-5885000.0
2008,-102165000.0


In [ ]:
df

,Country.Name,Country.Code,Indicator.Code,Indicator.Name,1960,1961,1962,1963,1964,1965,...,2021,2022,2023,outlier years,Data_error_1,Comments,Data_error_2,Comment_2,Data_error_3,Unnamed: 75
1,Algeria,DZA,DT.NFL.MIBR.CD,"Net financial flows, IBRD (NFL, current US$)",NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2006,True,Why would net financial flows fall from 49 mn ...,NaN,NaN,NaN,NaN
2,Angola,AGO,DC.DAC.FRAL.CD,"Net bilateral aid flows from DAC donors, Franc...",NaN,NaN,NaN,NaN,NaN,NaN,...,2.970000e+07,2.925000e+07,NaN,2006,True,All other numbers in the series are positive,NaN,NaN,NaN,NaN
3,Aruba,ABW,RQ.STD.ERR,Regulatory Quality: Standard Error,NaN,NaN,NaN,NaN,NaN,NaN,...,3.791812e-01,3.632358e-01,NaN,2005,False,This is an indication of statistical confidenc...,NaN,NaN,NaN,NaN
4,Bhutan,BTN,NY.GDP.DEFL.KD.ZG,"Inflation, GDP deflator (annual %)",NaN,NaN,NaN,NaN,NaN,NaN,...,7.628020e+00,5.794860e+00,NaN,1978,True,15 percent reported deflation in 1978 seems od...,NaN,NaN,NaN,NaN
5,Bolivia,BOL,FM.AST.CGOV.ZG.M3,Claims on central government (annual growth as...,NaN,10.241657,5.768856,2.992476,3.306727,10.530019,...,4.793785e+00,3.695174e+00,NaN,1985,False,There was severe hyperinflation in Bolivia at ...,NaN,NaN,NaN,NaN
6,Cameroon,CMR,DT.NFL.DPNG.CD,"Net flows on external debt, private nonguarant...",NaN,NaN,NaN,NaN,NaN,NaN,...,-1.124620e+09,-6.631000e+07,NaN,2021,False,France suspended XAF 197.2 billion of debt in ...,NaN,NaN,NaN,NaN
7,Canada,CAN,SH.TBS.DTEC.ZS,"Tuberculosis case detection rate (%, all forms)",NaN,NaN,NaN,NaN,NaN,NaN,...,8.700000e+01,9.000000e+01,NaN,2022,False,WHO estimates. But no change from 2000-2021 an...,NaN,NaN,NaN,NaN
8,China,CHN,SM.POP.REFG,Refugee population by country or territory of ...,NaN,NaN,NaN,NaN,NaN,NaN,...,3.034360e+05,3.200000e+02,2.960000e+02,1966 2022 2023,True,Why is there a reported figure for 1966? And w...,True,Less than 400 refugees in China seems wrong,True,Less than 400 refugees in China seems wrong
9,Croatia,HRV,SH.TBS.DTEC.ZS,"Tuberculosis case detection rate (%, all forms)",NaN,NaN,NaN,NaN,NaN,NaN,...,8.700000e+01,1.900000e+02,NaN,2022,True,Case detection rate cannot exceed 100%,NaN,NaN,NaN,NaN
10,Cyprus,CYP,SH.TBS.DTEC.ZS,"Tuberculosis case detection rate (%, all forms)",NaN,NaN,NaN,NaN,NaN,NaN,...,8.700000e+01,9.100000e+01,NaN,2022,False,WHO estimates. But no change from 2000-2021 an...,NaN,NaN,NaN,NaN


In [ ]:
b = 128
gen_batch = 30
inference_batch = []
output_cols=["cons_ppp17", "log_cons_ppp17"]
device = "cuda:1"

for g, g_df in tqdm(val_df.groupby("survey_id"), position=0):
    g_df = g_df[[i for i in g_df.columns if i not in output_cols]]

    for ix in tqdm(range(0, g_df.shape[0], b), position=1, leave=False):
        o = g_df.iloc[ix: ix + b]
        sd = sampler.sample_tabular_with_seed(seed_input=o, gen_batch=gen_batch, device=device, output_cols=output_cols)
        sd["hhid"] = sum([[i] * gen_batch for i in o["hhid"]], [])

        inference_batch.append(sd)

# out_fname = "preds-" + model_id.lstrip(".").replace("/", "__").strip("__")
# out_fname += ".csv"

inference_batch_df = pd.concat(inference_batch)
# inference_batch_df.to_csv(out_fname, index=None)

from scipy.stats import kruskal

groups = [
    val_preds["log_cons_ppp17"].values
]

groups.extend([
    val_df["log_cons_ppp17"].values
])

stat, p = kruskal(*groups)
stat, p